# Milestone 10: Deep Learning (CNN) Benchmark on PatchCamelyon (PCam)
**ES 335: Machine Learning Course Project - IIT Gandhinagar**
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

### Objectives:
1. Break through the classical feature representation ceiling (~79.8% test accuracy) using deep Convolutional Neural Networks.
2. Fine-tune **ResNet-18** with rotational invariant histology augmentations.
3. Perform one-shot evaluation on the untouched **5,000 held-out test cohort**.
4. Export predictions for consensus autopsy comparison against the 613 classical failures.

In [ ]:
# 1. Environment & GPU Verification
import torch
import torchvision
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('PyTorch version:    ', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('CUDA Available:     ', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('Device:             ', torch.cuda.get_device_name(0))
    print('Total GPU VRAM:     ', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print('WARNING: Running on CPU. Enable GPU under Settings -> Accelerator.')

In [ ]:
# 2. Fast Dataset Definition & Pathology Transforms (Instant RAM Linking)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class PCamDataset(Dataset):
    def __init__(self, split_dir, labels_csv, transform=None):
        self.df = pd.read_csv(labels_csv)
        self.transform = transform
        
        # Pure RAM path string join (takes 0.05s instead of 10 minutes over network disk)
        self.img_paths = [
            f"{split_dir}/{f}" if f.startswith('images') else f"{split_dir}/images/{f}"
            for f in self.df['filename'].values
        ]
        self.labels = self.df['label'].values.astype(np.float32)
        self.ids = [str(x) for x in self.df.get('id', range(len(self.df)))]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx], self.ids[idx], self.img_paths[idx]

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomChoice([
        transforms.RandomRotation((0, 0)),
        transforms.RandomRotation((90, 90)),
        transforms.RandomRotation((180, 180)),
        transforms.RandomRotation((270, 270)),
    ]),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    normalize,
])

eval_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    normalize,
])
print('Dataset class and transforms compiled successfully!')

In [ ]:
# 3. Instant Folder Discovery & DataLoader Initialization
data_dir = None

# Priority direct candidates including Kaggle dataset mount paths
candidates = [
    '/kaggle/input/datasets/pateldevarshh28/pcam-data',
    '/kaggle/input/datasets/pateldevarshh28/pcam-data/pcam_data',
    '/kaggle/input/pcam-data',
    '/kaggle/input/pcam-data/pcam_data',
]
for c in candidates:
    if os.path.exists(os.path.join(c, 'train')):
        data_dir = c
        break

# Deep recursive auto-detection if mounted at another nested level
if data_dir is None and os.path.exists('/kaggle/input'):
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'train' in dirs and 'test' in dirs:
            data_dir = root
            break

# Local environment fallback
if data_dir is None:
    for candidate in ['../data', 'data', 'pcam_tumor_detection/data']:
        if os.path.exists(os.path.join(candidate, 'train')):
            data_dir = os.path.abspath(candidate)
            break

if data_dir is None:
    raise FileNotFoundError('Dataset not found! Verify pcam-data is attached under Inputs.')

output_dir = '/kaggle/working/outputs/cnn' if os.path.exists('/kaggle') else os.path.abspath('../outputs/cnn')
os.makedirs(output_dir, exist_ok=True)
print('Data Directory located at:', data_dir)

train_dataset = PCamDataset(os.path.join(data_dir, 'train'), os.path.join(data_dir, 'train', 'train_labels.csv'), transform=train_transform)
val_dataset   = PCamDataset(os.path.join(data_dir, 'validation'), os.path.join(data_dir, 'validation', 'validation_labels.csv'), transform=eval_transform)
test_dataset  = PCamDataset(os.path.join(data_dir, 'test'), os.path.join(data_dir, 'test', 'test_labels.csv'), transform=eval_transform)

print('Loaded splits: Train=' + str(len(train_dataset)) + ' | Val=' + str(len(val_dataset)) + ' | Test=' + str(len(test_dataset)))

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('DataLoaders ready with batch size', BATCH_SIZE)


In [ ]:
# 4. Initialize Pretrained ResNet-18 Model
import torch.nn as nn
import torchvision.models as models

print('Loading ResNet-18 pretrained weights...')
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = model.fc.in_features

# Custom binary classification head
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 1)
)
model = model.to(device)
print('Model successfully loaded onto', device)

In [ ]:
# 5. Training Loop with Live Progress Bar & Mixed Precision
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

EPOCHS = 10
LR = 2e-4

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_val_auc = 0.0
checkpoint_path = os.path.join(output_dir, 'best_resnet18_notebook.pt')

print('=' * 75)
print(f"STARTING TRAINING: RESNET-18 ({EPOCHS} EPOCHS, BATCH {BATCH_SIZE})")
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]")
    for images, targets, _, _ in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.float().unsqueeze(1).to(device, non_blocking=True)
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs).detach().cpu().numpy().flatten()
        all_preds.extend(probs)
        all_targets.extend(targets.cpu().numpy().flatten())
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    tr_loss = running_loss / len(train_dataset)
    tr_auc  = roc_auc_score(all_targets, all_preds)
    
    # Validation pass
    model.eval()
    val_loss_run = 0.0
    v_preds, v_targets = [], []
    with torch.no_grad():
        for images, targets, _, _ in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.float().unsqueeze(1).to(device, non_blocking=True)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, targets)
            val_loss_run += loss.item() * images.size(0)
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            v_preds.extend(probs)
            v_targets.extend(targets.cpu().numpy().flatten())
            
    vl_loss = val_loss_run / len(val_dataset)
    vl_acc  = accuracy_score(v_targets, (np.array(v_preds) >= 0.5).astype(int))
    vl_auc  = roc_auc_score(v_targets, v_preds)
    
    print(f"--> Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {tr_loss:.4f} | Train AUC: {tr_auc:.4f} || Val Acc: {vl_acc*100:6.2f}% | Val AUC: {vl_auc:.4f}")
    
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        torch.save(model.state_dict(), checkpoint_path)
        print(f"    * Saved new best model checkpoint (Val AUC: {vl_auc:.4f})")

print(f"\n[Training Done] Peak Validation ROC-AUC: {best_val_auc:.4f}")

In [ ]:
# 6. One-Shot Test Cohort Evaluation (Exact Held-Out N = 5,000 Patches)
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc, brier_score_loss

print('Loading best checkpoint for sealed test evaluation...')
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

t_preds, t_targets, t_ids, t_paths = [], [], [], []
with torch.no_grad():
    for images, targets, p_ids, p_paths in tqdm(test_loader, desc='Evaluating Test Patches'):
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        t_preds.extend(probs)
        t_targets.extend(targets.numpy().flatten())
        t_ids.extend(p_ids)
        t_paths.extend(p_paths)

t_preds   = np.array(t_preds)
t_targets = np.array(t_targets).astype(int)
t_binary  = (t_preds >= 0.5).astype(int)

test_acc   = accuracy_score(t_targets, t_binary)
test_auc   = roc_auc_score(t_targets, t_preds)
prec_c, rec_c, _ = precision_recall_curve(t_targets, t_preds)
test_pr    = auc(rec_c, prec_c)
cm         = confusion_matrix(t_targets, t_binary)
tn, fp, fn, tp = cm.ravel()
sens       = tp / (tp + fn)
spec       = tn / (tn + fp)
prec       = tp / (tp + fp) if (tp + fp) > 0 else 0.0
f1         = 2 * (prec * sens) / (prec + sens) if (prec + sens) > 0 else 0.0
brier      = brier_score_loss(t_targets, t_preds)

print('\n' + '=' * 60)
print(' FINAL PCAM TEST COHORT EVALUATION (RESNET-18)')
print('=' * 60)
print(f" Test Accuracy:     {test_acc * 100:.2f}%")
print(f" Test ROC-AUC:      {test_auc:.4f}")
print(f" Test PR-AUC:       {test_pr:.4f}")
print(f" Sensitivity:       {sens * 100:.2f}%")
print(f" Specificity:       {spec * 100:.2f}%")
print(f" Precision:         {prec * 100:.2f}%")
print(f" F1-Score:          {f1:.4f}")
print(f" Brier Score:       {brier:.4f}")
print('-' * 60)
print(f" Confusion Matrix (N = 5,000): TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")
print('=' * 60)

# Save predictions CSV for error autopsy
test_df = pd.DataFrame({
    'id': t_ids,
    'filename': t_paths,
    'y_true': t_targets,
    'y_pred': t_binary,
    'prob_tumor': np.round(t_preds, 5)
})

csv_out_path = os.path.join(output_dir, 'test_predictions_resnet18.csv')
test_df.to_csv(csv_out_path, index=False)
if os.path.exists('/kaggle/working'):
    test_df.to_csv('/kaggle/working/test_predictions_resnet18.csv', index=False)

print(f"[Saved] Test predictions exported to: test_predictions_resnet18.csv")

In [ ]:
# 7. Direct Scientific Benchmark Leaderboard (Classical vs. Deep Learning)
results_table = pd.DataFrame([
    {'Architecture': 'Logistic Regression (Baseline)', 'Test Accuracy': '75.84%', 'ROC-AUC': 0.8373, 'Sensitivity': '70.96%', 'Precision': '78.63%', 'Paradigm': 'Linear Baseline'},
    {'Architecture': 'Linear SVM (Dual C=1.0)',        'Test Accuracy': '75.46%', 'ROC-AUC': 0.8292, 'Sensitivity': '71.12%', 'Precision': '77.88%', 'Paradigm': 'Linear Baseline'},
    {'Architecture': 'Tuned Random Forest (150)',     'Test Accuracy': '78.46%', 'ROC-AUC': 0.8863, 'Sensitivity': '68.76%', 'Precision': '85.31%', 'Paradigm': 'Tree Ensemble'},
    {'Architecture': 'HistGradientBoosting',          'Test Accuracy': '78.82%', 'ROC-AUC': 0.8845, 'Sensitivity': '71.84%', 'Precision': '83.50%', 'Paradigm': 'Gradient Boosting'},
    {'Architecture': 'RBF Kernel SVM (Non-linear)',   'Test Accuracy': '78.80%', 'ROC-AUC': 0.8746, 'Sensitivity': '73.28%', 'Precision': '82.37%', 'Paradigm': 'Kernel Machine'},
    {'Architecture': 'Hybrid Blend (50/50 RBF+Hist)', 'Test Accuracy': '79.76%', 'ROC-AUC': 0.8853, 'Sensitivity': '73.08%', 'Precision': '84.35%', 'Paradigm': 'Classical Champion'},
    {'Architecture': 'ResNet-18 (Deep CNN)',          'Test Accuracy': f"{test_acc * 100:.2f}%", 'ROC-AUC': round(test_auc, 4), 'Sensitivity': f"{sens * 100:.2f}%", 'Precision': f"{prec * 100:.2f}%", 'Paradigm': 'Deep Learning'}
])

display(results_table)